# Exercise solutions

Working answers to every "Try it" exercise, grouped by lesson. Try each exercise yourself before looking.

## Lesson 00 · Setup and check

**1.** Add `"langgraph-checkpoint"` to the list and print its version.

In [1]:
from importlib.metadata import version

for name in ["langgraph", "langchain", "langgraph-checkpoint"]:
    print(f"{name:30} {version(name)}")

langgraph                      1.2.12
langchain                      1.4.3
langgraph-checkpoint           4.2.0


## Lesson 01 · What is a graph?

**1.** Add `**` (power) to `calculate`. The nodes from lesson 01 are repeated here so this cell runs on its own.

In [2]:
def parse(state):
    left, op, right = state["expression"].split()
    return {"a": float(left), "op": op, "b": float(right)}


def calculate(state):
    a, op, b = state["a"], state["op"], state["b"]
    if op == "+":
        result = a + b
    elif op == "-":
        result = a - b
    elif op == "*":
        result = a * b
    elif op == "/":
        result = a / b
    elif op == "**":  # new
        result = a ** b
    else:
        raise ValueError(f"Unknown operator: {op}")
    return {"result": result}

In [3]:
def format_answer(state):
    return {"answer": f"{state['expression']} = {state['result']:g}"}


steps = [parse, calculate, format_answer]


def run_calculator(expression):
    state = {"expression": expression}
    for step in steps:
        state.update(step(state))
    return state


print(run_calculator("2 ** 10")["answer"])

2 ** 10 = 1024


**2.** A new node, `round_result`, goes between `calculate` and `format_answer`. Adding a step is just adding a
function and putting it in the list of edges.

In [4]:
def round_result(state):
    return {"result": round(state["result"], 2)}


steps = [parse, calculate, round_result, format_answer]
print(run_calculator("10 / 3")["answer"])

10 / 3 = 3.33


## Lesson 02 · State

**1.** `CalcState2` adds an `op` key.

In [5]:
from typing import TypedDict


class CalcState2(TypedDict):
    a: float
    b: float
    op: str
    result: float


state: CalcState2 = {"a": 6.0, "b": 7.0, "op": "*", "result": 0.0}
print(f"{state['a']} {state['op']} {state['b']}")

6.0 * 7.0


**2.** `b="0.5"` is text that can be safely read as a number, so Pydantic converts it to `0.5`. `None` is not a
number, so Pydantic rejects it with a `ValidationError`.

In [6]:
from pydantic import BaseModel, ValidationError


class CalcStateModel(BaseModel):
    a: float
    b: float
    result: float = 0.0


print(CalcStateModel(a=3, b="0.5"))

try:
    CalcStateModel(a=3, b=None)
except ValidationError as err:
    print(err)

a=3.0 b=0.5 result=0.0
1 validation error for CalcStateModel
b
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/float_type


## Lesson 03 · Nodes

**1.** A multiply node, called directly.

In [7]:
from typing import TypedDict

from langgraph.graph import StateGraph


class CalcState(TypedDict):
    a: float
    b: float
    result: float


def multiply_node_fn(state: CalcState) -> dict:
    return {"result": state["a"] * state["b"]}


print(multiply_node_fn({"a": 6.0, "b": 7.0}))

{'result': 42.0}


**2.** Registering it works. Registering a second node called `"add"` fails straight away, because node names
must be unique.

In [8]:
def add_node_fn(state: CalcState) -> dict:
    return {"result": state["a"] + state["b"]}


builder = StateGraph(CalcState)
builder.add_node("add", add_node_fn)
builder.add_node("multiply", multiply_node_fn)
print(list(builder.nodes))

try:
    builder.add_node("add", add_node_fn)
except ValueError as err:
    print("ValueError:", err)

['add', 'multiply']
ValueError: Node `add` already present.


## Lesson 04 · Edges, START, END, compile and invoke

**1.** `square` runs after `add`, so `(3 + 4) * (3 + 4) = 49`.

In [9]:
from langgraph.graph import START, END


def square(state: CalcState) -> dict:
    return {"result": state["result"] * state["result"]}


builder = StateGraph(CalcState)
builder.add_node("add", add_node_fn)
builder.add_node("square", square)
builder.add_edge(START, "add")
builder.add_edge("add", "square")
builder.add_edge("square", END)

graph = builder.compile()
print(graph.get_graph().draw_mermaid())
print(graph.invoke({"a": 3.0, "b": 4.0}))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	add(add)
	square(square)
	__end__([<p>__end__</p>]):::last
	__start__ --> add;
	add --> square;
	square --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

{'a': 3.0, 'b': 4.0, 'result': 49.0}


**2.** Only `parse` runs. `calculate` and `format` are still registered, but no edge leads to them, so they never
run. LangGraph allows this (it doesn't complain about unreachable nodes). The final state has `a`, `op` and `b` but
no `result` or `answer`.

In [10]:
class PipelineState(TypedDict):
    expression: str
    a: float
    op: str
    b: float
    result: float
    answer: str


def parse(state: PipelineState) -> dict:
    left, op, right = state["expression"].split()
    return {"a": float(left), "op": op, "b": float(right)}

In [11]:
def calculate(state: PipelineState) -> dict:
    return {"result": state["a"] + state["b"]}  # simplified: it never runs here anyway


def format_answer(state: PipelineState) -> dict:
    return {"answer": f"{state['expression']} = {state['result']:g}"}


builder = StateGraph(PipelineState)
builder.add_node("parse", parse)
builder.add_node("calculate", calculate)
builder.add_node("format", format_answer)
builder.add_edge(START, "parse")
builder.add_edge("parse", END)  # was: parse -> calculate
builder.add_edge("calculate", "format")
builder.add_edge("format", END)

print(builder.compile().invoke({"expression": "3 + 4"}))

{'expression': '3 + 4', 'a': 3.0, 'op': '+', 'b': 4.0}


## Lesson 05 · Conditional edges

Both answers use the lesson 05 state and nodes, repeated here so the cells run on their own.

In [12]:
from typing import Literal


class CalcState5(TypedDict):
    expression: str
    a: float
    op: str
    b: float
    result: float
    answer: str


def add(state):
    return {"result": state["a"] + state["b"]}


def subtract(state):
    return {"result": state["a"] - state["b"]}

In [13]:
def multiply(state):
    return {"result": state["a"] * state["b"]}


def divide(state):
    return {"result": state["a"] / state["b"]}


def format_answer5(state):
    return {"answer": f"{state['expression']} = {state['result']:g}"}


def error(state):
    return {"answer": f"Sorry, I don't know the operator {state['op']!r}. Use +, -, * or /."}

**1.** A `divide_by_zero` node. The router checks for `/` with `b == 0` *before* the normal lookup, and the new
name goes in the `Literal` hint.

In [14]:
def divide_by_zero(state):
    return {"answer": f"{state['expression']}: can't divide by zero"}


OPERATIONS = {"+": "add", "-": "subtract", "*": "multiply", "/": "divide"}


def route(state) -> Literal["add", "subtract", "multiply", "divide", "divide_by_zero", "error"]:
    if state["op"] == "/" and state["b"] == 0:
        return "divide_by_zero"
    return OPERATIONS.get(state["op"], "error")

In [15]:
builder = StateGraph(CalcState5)
builder.add_node("parse", parse)
for name, fn in [("add", add), ("subtract", subtract), ("multiply", multiply), ("divide", divide)]:
    builder.add_node(name, fn)
    builder.add_edge(name, "format")
builder.add_node("format", format_answer5)
builder.add_node("error", error)
builder.add_node("divide_by_zero", divide_by_zero)
builder.add_edge(START, "parse")
builder.add_conditional_edges("parse", route)
builder.add_edge("format", END)
builder.add_edge("error", END)
builder.add_edge("divide_by_zero", END)

graph = builder.compile()
for expression in ["10 / 0", "10 / 4", "2 % 3"]:
    print(graph.invoke({"expression": expression})["answer"])

10 / 0: can't divide by zero
10 / 4 = 2.5
Sorry, I don't know the operator '%'. Use +, -, * or /.


**2.** A `modulo` node for `%`. Three changes: the node, the `OPERATIONS` entry and the `Literal` name.

In [16]:
def modulo(state):
    return {"result": state["a"] % state["b"]}


OPERATIONS = {"+": "add", "-": "subtract", "*": "multiply", "/": "divide", "%": "modulo"}


def route(state) -> Literal["add", "subtract", "multiply", "divide", "modulo", "error"]:
    return OPERATIONS.get(state["op"], "error")

In [17]:
builder = StateGraph(CalcState5)
builder.add_node("parse", parse)
for name, fn in [("add", add), ("subtract", subtract), ("multiply", multiply), ("divide", divide), ("modulo", modulo)]:
    builder.add_node(name, fn)
    builder.add_edge(name, "format")
builder.add_node("format", format_answer5)
builder.add_node("error", error)
builder.add_edge(START, "parse")
builder.add_conditional_edges("parse", route)
builder.add_edge("format", END)
builder.add_edge("error", END)

graph = builder.compile()
print(graph.get_graph().draw_mermaid())
print(graph.invoke({"expression": "17 % 5"})["answer"])

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	add(add)
	subtract(subtract)
	multiply(multiply)
	divide(divide)
	modulo(modulo)
	format(format)
	error(error)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	add --> format;
	divide --> format;
	modulo --> format;
	multiply --> format;
	parse -.-> add;
	parse -.-> divide;
	parse -.-> error;
	parse -.-> modulo;
	parse -.-> multiply;
	parse -.-> subtract;
	subtract --> format;
	error --> __end__;
	format --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

17 % 5 = 2


## Lesson 06 · Reducers

**1.** `steps` ends at `2`. The input gives `0`, and each node returns `1`. The `operator.add` reducer *adds*
numbers, so the total is `0 + 1 + 1`. It counts how many nodes ran.

In [18]:
import operator
from typing import Annotated, Literal, TypedDict

from langgraph.graph import StateGraph, START, END


class DemoState(TypedDict):
    last: str
    history: Annotated[list[str], operator.add]
    steps: Annotated[int, operator.add]


def first(state: DemoState) -> dict:
    return {"last": "first", "history": ["first"], "steps": 1}


def second(state: DemoState) -> dict:
    return {"last": "second", "history": ["second"], "steps": 1}

In [19]:
builder = StateGraph(DemoState)
builder.add_node("first", first)
builder.add_node("second", second)
builder.add_edge(START, "first")
builder.add_edge("first", "second")
builder.add_edge("second", END)
demo = builder.compile()

print(demo.invoke({"steps": 0}))

{'last': 'second', 'history': ['first', 'second'], 'steps': 2}


**2.** `no_duplicates` starts from the old list and appends each new item only if it isn't there yet.

In [20]:
def no_duplicates(old: list[str], new: list[str]) -> list[str]:
    merged = list(old)
    for item in new:
        if item not in merged:
            merged.append(item)
    return merged


print(no_duplicates(["add: 7"], ["add: 7", "multiply: 42"]))

['add: 7', 'multiply: 42']


## Lesson 07 · Loops and the recursion limit

**1.** Halving 1000 takes 10 rounds and ends at `0.9765625` (the first value below 1).

In [21]:
class HalveState(TypedDict):
    value: float
    rounds: int


def halve(state: HalveState) -> dict:
    return {"value": state["value"] / 2, "rounds": state["rounds"] + 1}


def below_one(state: HalveState) -> Literal["halve", END]:
    return END if state["value"] < 1 else "halve"


builder = StateGraph(HalveState)
builder.add_node("halve", halve)
builder.add_edge(START, "halve")
builder.add_conditional_edges("halve", below_one)
print(builder.compile().invoke({"value": 1000.0, "rounds": 0}))

{'value': 0.9765625, 'rounds': 10}


**2.** Starting from `1e-30`, it takes about 107 doublings to get past 100. That's more than a limit of 50
allows, so the run stops with `GraphRecursionError` even though the loop is correct. A limit with headroom, such as
200, lets it finish.

In [22]:
from langgraph.errors import GraphRecursionError


class DoubleState(TypedDict):
    value: float
    rounds: int


def double(state: DoubleState) -> dict:
    return {"value": state["value"] * 2, "rounds": state["rounds"] + 1}


def should_continue(state: DoubleState) -> Literal["double", END]:
    return END if state["value"] > 100 else "double"

In [23]:
builder = StateGraph(DoubleState)
builder.add_node("double", double)
builder.add_edge(START, "double")
builder.add_conditional_edges("double", should_continue)
doubler = builder.compile()

try:
    doubler.invoke({"value": 1e-30, "rounds": 0}, {"recursion_limit": 50})
except GraphRecursionError as err:
    print("limit 50:  GraphRecursionError:", str(err).splitlines()[0])

print("limit 200:", doubler.invoke({"value": 1e-30, "rounds": 0}, {"recursion_limit": 200}))

limit 50:  GraphRecursionError: Recursion limit of 50 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.


limit 200: {'value': 162.25927682921338, 'rounds': 107}


## Lesson 08 · Parallel branches

**1.** A fifth branch is one more node and two more edges.

In [24]:
class ParallelState(TypedDict):
    a: float
    b: float
    results: Annotated[list[str], operator.add]
    summary: str


def make_branch(symbol, fn):
    def node(state: ParallelState) -> dict:
        return {"results": [f"{state['a']:g} {symbol} {state['b']:g} = {fn(state['a'], state['b']):g}"]}

    return node


def summary(state: ParallelState) -> dict:
    return {"summary": " | ".join(state["results"])}

In [25]:
branches = {
    "add": make_branch("+", operator.add),
    "subtract": make_branch("-", operator.sub),
    "multiply": make_branch("*", operator.mul),
    "divide": make_branch("/", operator.truediv),
    "power": make_branch("**", operator.pow),  # new
}

builder = StateGraph(ParallelState)
builder.add_node("start_node", lambda state: {})
builder.add_node("summary", summary)
for name, fn in branches.items():
    builder.add_node(name, fn)
    builder.add_edge("start_node", name)
    builder.add_edge(name, "summary")
builder.add_edge(START, "start_node")
builder.add_edge("summary", END)

print(builder.compile().invoke({"a": 12, "b": 4})["summary"])

12 + 4 = 16 | 12 / 4 = 3 | 12 * 4 = 48 | 12 ** 4 = 20736 | 12 - 4 = 8


**2.** With `operator.add` on a number, each branch's number is *added* to the total: `16 + 8 + 48 + 3 = 75`.

In [26]:
class TotalState(TypedDict):
    a: float
    b: float
    total: Annotated[float, operator.add]


def number_branch(fn):
    return lambda state: {"total": fn(state["a"], state["b"])}


builder = StateGraph(TotalState)
builder.add_node("start_node", lambda state: {})
for name, fn in [("add", operator.add), ("subtract", operator.sub), ("multiply", operator.mul), ("divide", operator.truediv)]:
    builder.add_node(name, number_branch(fn))
    builder.add_edge("start_node", name)
    builder.add_edge(name, END)
builder.add_edge(START, "start_node")

print(builder.compile().invoke({"a": 12, "b": 4, "total": 0}))

{'a': 12, 'b': 4, 'total': 75.0}


## Lesson 09 · Map-reduce with `Send`

**1.** `total` also stores the largest result with `max`.

In [27]:
import operator
from typing import Annotated, Literal, TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, Send

OPERATIONS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}


def calc(expr: str) -> float:
    for symbol, fn in OPERATIONS.items():
        if symbol in expr:
            left, right = expr.split(symbol)
            return fn(float(left), float(right))
    raise ValueError(f"No operator found in {expr!r}")

In [28]:
class MapReduceState(TypedDict):
    exprs: list[str]
    results: Annotated[list[float], operator.add]
    total: float
    largest: float


def evaluate(state: dict) -> dict:
    return {"results": [calc(state["expr"])]}


def total(state: MapReduceState) -> dict:
    return {"total": sum(state["results"]), "largest": max(state["results"])}


builder = StateGraph(MapReduceState)
builder.add_node("evaluate", evaluate)
builder.add_node("total", total)
builder.add_conditional_edges(START, lambda s: [Send("evaluate", {"expr": e}) for e in s["exprs"]], ["evaluate"])
builder.add_edge("evaluate", "total")
builder.add_edge("total", END)
print(builder.compile().invoke({"exprs": ["1+2", "3*4", "10/5"]}))

{'exprs': ['1+2', '3*4', '10/5'], 'results': [3.0, 12.0, 2.0], 'total': 17.0, 'largest': 12.0}


**2.** The payload can carry anything the node needs, such as the position. We sort the lines by position at the
end, because the order of parallel results isn't guaranteed. (Here the position is the first number in each line,
so we sort on that.)

In [29]:
class LinesState(TypedDict):
    exprs: list[str]
    lines: Annotated[list[str], operator.add]


def evaluate_line(state: dict) -> dict:
    value = calc(state["expr"])
    return {"lines": [f"#{state['position']}: {state['expr']} = {value:g}"]}


def fan_out(state: LinesState) -> list[Send]:
    return [Send("evaluate", {"expr": expr, "position": i}) for i, expr in enumerate(state["exprs"], start=1)]


builder = StateGraph(LinesState)
builder.add_node("evaluate", evaluate_line)
builder.add_conditional_edges(START, fan_out, ["evaluate"])
builder.add_edge("evaluate", END)
out = builder.compile().invoke({"exprs": ["1+2", "3*4", "10/5"]})
for line in sorted(out["lines"], key=lambda line: int(line[1:line.index(":")])):
    print(line)

#1: 1+2 = 3
#2: 3*4 = 12
#3: 10/5 = 2


## Lesson 10 · `Command`

The lesson 10 state and nodes, repeated so the cells run on their own.

In [30]:
class CalcState(TypedDict):
    expression: str
    a: float
    op: str
    b: float
    result: float
    answer: str


def add(state):
    return {"result": state["a"] + state["b"]}


def subtract(state):
    return {"result": state["a"] - state["b"]}


def multiply(state):
    return {"result": state["a"] * state["b"]}


def divide(state):
    return {"result": state["a"] / state["b"]}

In [31]:
def format_answer(state):
    return {"answer": f"{state['expression']} = {state['result']:g}"}


def error(state) -> Command[Literal[END]]:
    return Command(update={"answer": f"Sorry, I don't know the operator {state['op']!r}."}, goto=END)


OP_NODES = {"+": "add", "-": "subtract", "*": "multiply", "/": "divide"}

**1.** `parse` checks for division by zero before choosing the normal route. The new node goes in the hint.

In [32]:
def parse(state) -> Command[Literal["add", "subtract", "multiply", "divide", "divide_by_zero", "error"]]:
    left, op, right = state["expression"].split()
    a, b = float(left), float(right)
    if op == "/" and b == 0:
        goto = "divide_by_zero"
    else:
        goto = OP_NODES.get(op, "error")
    return Command(update={"a": a, "op": op, "b": b}, goto=goto)


def divide_by_zero(state) -> Command[Literal[END]]:
    return Command(update={"answer": f"{state['expression']}: can't divide by zero"}, goto=END)

In [33]:
builder = StateGraph(CalcState)
builder.add_node("parse", parse)
for name, fn in [("add", add), ("subtract", subtract), ("multiply", multiply), ("divide", divide)]:
    builder.add_node(name, fn)
    builder.add_edge(name, "format")
builder.add_node("format", format_answer)
builder.add_node("error", error)
builder.add_node("divide_by_zero", divide_by_zero)
builder.add_edge(START, "parse")
builder.add_edge("format", END)

graph = builder.compile()
for expression in ["10 / 0", "10 / 4", "2 % 3"]:
    print(graph.invoke({"expression": expression})["answer"])

10 / 0: can't divide by zero
10 / 4 = 2.5
Sorry, I don't know the operator '%'.


**2.** Yes, it still works. The diagram now shows a **dashed** `format -.-> __end__`, because the route comes
from a `Command` instead of a fixed edge.

In [34]:
def format_command(state) -> Command[Literal[END]]:
    return Command(update={"answer": f"{state['expression']} = {state['result']:g}"}, goto=END)


builder = StateGraph(CalcState)
builder.add_node("parse", parse)
for name, fn in [("add", add), ("subtract", subtract), ("multiply", multiply), ("divide", divide)]:
    builder.add_node(name, fn)
    builder.add_edge(name, "format")
builder.add_node("format", format_command)
builder.add_node("error", error)
builder.add_node("divide_by_zero", divide_by_zero)
builder.add_edge(START, "parse")  # no add_edge("format", END) any more

graph = builder.compile()
print(graph.invoke({"expression": "6 * 7"})["answer"])
print([line.strip() for line in graph.get_graph().draw_mermaid().splitlines() if line.strip().startswith("format ")])

6 * 7 = 42
['format -.-> __end__;']


## Lesson 11 · Input, output and private state schemas

**1.** Only the output schema changes: add `result` to it. `OverallState` inherits it, so nothing else changes.

In [35]:
class InputState(TypedDict):
    expression: str


class OutputState(TypedDict):
    answer: str
    result: float  # new


class OverallState(InputState, OutputState):
    a: float
    op: str
    b: float

In [36]:
def parse11(state: OverallState) -> dict:
    left, op, right = state["expression"].split()
    return {"a": float(left), "op": op, "b": float(right)}


def multiply11(state: OverallState) -> dict:  # only * is needed for this check
    return {"result": state["a"] * state["b"]}


def format11(state: OverallState) -> dict:
    return {"answer": f"{state['expression']} = {state['result']:g}"}

In [37]:
def build11(input_schema, output_schema):
    builder = StateGraph(OverallState, input_schema=input_schema, output_schema=output_schema)
    builder.add_node("parse", parse11)
    builder.add_node("calculate", multiply11)
    builder.add_node("format", format11)
    builder.add_edge(START, "parse")
    builder.add_edge("parse", "calculate")
    builder.add_edge("calculate", "format")
    builder.add_edge("format", END)
    return builder.compile()


print(build11(InputState, OutputState).invoke({"expression": "6 * 7"}))

{'answer': '6 * 7 = 42', 'result': 42.0}


**2.** `request_id` goes in **both** `InputState` (so it's accepted) and `OutputState` (so it's returned).
`OverallState` inherits it from them. No node writes it, so it passes through unchanged. We redefine the three
schemas, then reuse `build11`. Its nodes are typed with the old `OverallState`, so we redefine them too (lesson 06:
LangGraph reads node type hints).

In [38]:
class InputState(TypedDict):
    expression: str
    request_id: str


class OutputState(TypedDict):
    answer: str
    request_id: str


class OverallState(InputState, OutputState):
    a: float
    op: str
    b: float
    result: float

In [39]:
def parse11(state: OverallState) -> dict:
    left, op, right = state["expression"].split()
    return {"a": float(left), "op": op, "b": float(right)}


def multiply11(state: OverallState) -> dict:
    return {"result": state["a"] * state["b"]}


def format11(state: OverallState) -> dict:
    return {"answer": f"{state['expression']} = {state['result']:g}"}


print(build11(InputState, OutputState).invoke({"expression": "6 * 7", "request_id": "req-42"}))

{'answer': '6 * 7 = 42', 'request_id': 'req-42'}


## Lesson 12 · Runtime context

**1.** A second field on the context, read in the same node.

In [40]:
from dataclasses import dataclass
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.runtime import Runtime


@dataclass
class Context:
    decimal_places: int = 2
    show_expression: bool = True


class CalcState12(TypedDict):
    expression: str
    result: float
    answer: str

In [41]:
def calculate12(state: CalcState12) -> dict:
    left, op, right = state["expression"].split()
    a, b = float(left), float(right)
    if op == "+":
        return {"result": a + b}
    elif op == "-":
        return {"result": a - b}
    elif op == "*":
        return {"result": a * b}
    return {"result": a / b}


def format12(state: CalcState12, runtime: Runtime[Context]) -> dict:
    context = runtime.context or Context()
    number = f"{state['result']:.{context.decimal_places}f}"
    return {"answer": f"{state['expression']} = {number}" if context.show_expression else number}

In [42]:
builder = StateGraph(CalcState12, context_schema=Context)
builder.add_node("calculate", calculate12)
builder.add_node("format", format12)
builder.add_edge(START, "calculate")
builder.add_edge("calculate", "format")
builder.add_edge("format", END)
graph12 = builder.compile()

print(graph12.invoke({"expression": "10 / 3"}, context={"show_expression": True})["answer"])
print(graph12.invoke({"expression": "10 / 3"}, context={"show_expression": False})["answer"])

10 / 3 = 3.33
3.33


**2.** Ask "does it change during the run, or is it part of the answer?" If yes, it's state. If it's a setting
chosen before the run and only read, it's context.

| Item | Where | Why |
|---|---|---|
| The expression the user typed | State | It's the input data the nodes work on. |
| The user's preferred language | Context | A setting for the whole run; no node changes it. |
| The running total of a session | State | It changes every call (and a checkpointer remembers it, lesson 13). |
| The maximum number of retries | Context | A fixed setting that controls how the run behaves. |
| The `result` of a calculation | State | A node produces it during the run. |

## Lesson 13 · Checkpointers and threads

The lesson 13 state, node and in-memory graph, rebuilt so the cells run on their own.

In [43]:
import operator
from typing import Annotated

from langgraph.checkpoint.memory import InMemorySaver


class TotalState(TypedDict):
    amount: float
    total: float
    history: Annotated[list[str], operator.add]


def add_amount(state: TotalState) -> dict:
    new_total = state.get("total", 0.0) + state["amount"]
    return {"total": new_total, "history": [f"+{state['amount']:g} -> {new_total:g}"]}


builder = StateGraph(TotalState)
builder.add_node("add_amount", add_amount)
builder.add_edge(START, "add_amount")
builder.add_edge("add_amount", END)
graph13 = builder.compile(checkpointer=InMemorySaver())

**1.** The total is `6.0`. There are **9** checkpoints, because every `invoke` saves three (`input`, `loop` before
the node, `loop` after it), and Dave invoked three times.

In [44]:
dave = {"configurable": {"thread_id": "dave"}}
for amount in [1, 2, 3]:
    graph13.invoke({"amount": amount}, dave)

print("total:", graph13.get_state(dave).values["total"])
print("checkpoints:", len(list(graph13.get_state_history(dave))))

total: 6.0
checkpoints: 9


**2.** First we replay Alice's steps from the lesson (`+5`, `+10`, `+2.5`, the manual fix, `+1`). Then we go back to the
last checkpoint where the total was `17.5` and the run had finished, and add `50` from there. The new total is
`67.5`, and the `manual fix` line is **not** in the new branch's history: that branch starts before the fix.

In [45]:
alice = {"configurable": {"thread_id": "alice"}}
for amount in [5, 10, 2.5]:
    graph13.invoke({"amount": amount}, alice)
graph13.update_state(alice, {"total": 100.0, "history": ["manual fix: total set to 100"]})
graph13.invoke({"amount": 1}, alice)

before_fix = next(
    snap for snap in graph13.get_state_history(alice)
    if snap.values.get("total") == 17.5 and snap.next == ()
)
print(graph13.invoke({"amount": 50}, before_fix.config))

{'amount': 50, 'total': 67.5, 'history': ['+5 -> 5', '+10 -> 15', '+2.5 -> 17.5', '+50 -> 67.5']}


## Lesson 14 · Human-in-the-loop with `interrupt`

**1.** A `while` loop keeps asking. Each answer is matched to the next `interrupt` in order (Rule 4), and the node
re-runs from the top every time, so `divide started` prints **3** times: the first run, after the `0`, and after the
`4`. The result is `10 / 4 = 2.5`.

In [46]:
from typing import Literal

from langgraph.types import Command, interrupt


class DivideState(TypedDict):
    a: float
    b: float
    result: float


def divide_until_nonzero(state: DivideState) -> dict:
    print(f"  divide started (b = {state['b']})")
    b = state["b"]
    while b == 0:
        b = interrupt({"question": "b is 0. Enter a new divisor:"})
    return {"b": b, "result": state["a"] / b}

In [47]:
builder = StateGraph(DivideState)
builder.add_node("divide", divide_until_nonzero)
builder.add_edge(START, "divide")
builder.add_edge("divide", END)
graph14 = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "keep-asking"}}
print("asked:", graph14.invoke({"a": 10, "b": 0}, config)["__interrupt__"][0].value)
print("asked:", graph14.invoke(Command(resume=0), config)["__interrupt__"][0].value)
print("done: ", graph14.invoke(Command(resume=4), config))

  divide started (b = 0)
asked: {'question': 'b is 0. Enter a new divisor:'}
  divide started (b = 0)
asked: {'question': 'b is 0. Enter a new divisor:'}
  divide started (b = 0)
done:  {'a': 10, 'b': 4, 'result': 2.5}


**2.** The answer is now a dict. `approve` reads both fields and sends the new `b` on with `Command(update=...)`.
Ten divided by the corrected `2` is `5`.

In [48]:
class ApprovalState(TypedDict):
    a: float
    b: float
    answer: str


def approve(state: ApprovalState) -> Command[Literal["divide", "cancelled"]]:
    reply = interrupt({"question": f"Divide {state['a']:g} by {state['b']:g}? You may change b."})
    if not reply["approved"]:
        return Command(goto="cancelled")
    return Command(update={"b": reply.get("b", state["b"])}, goto="divide")


def divide_approved(state: ApprovalState) -> dict:
    return {"answer": f"{state['a']:g} / {state['b']:g} = {state['a'] / state['b']:g}"}


def cancelled(state: ApprovalState) -> dict:
    return {"answer": "Cancelled by the human."}

In [49]:
builder = StateGraph(ApprovalState)
builder.add_node("approve", approve)
builder.add_node("divide", divide_approved)
builder.add_node("cancelled", cancelled)
builder.add_edge(START, "approve")
builder.add_edge("divide", END)
builder.add_edge("cancelled", END)
approval = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "edit-b"}}
print("asked:", approval.invoke({"a": 10, "b": 4}, config)["__interrupt__"][0].value["question"])
print(approval.invoke(Command(resume={"approved": True, "b": 2}), config)["answer"])

asked: Divide 10 by 4? You may change b.
10 / 2 = 5


## Lesson 15 · Streaming

The lesson 15 calculator, rebuilt with a `divide` that also sends a progress message (exercise 1).

In [50]:
from typing import Literal, TypedDict

from langgraph.config import get_stream_writer
from langgraph.graph import StateGraph, START, END


class CalcState15(TypedDict):
    expression: str
    a: float
    op: str
    b: float
    result: float
    answer: str


def parse15(state):
    left, op, right = state["expression"].split()
    return {"a": float(left), "op": op, "b": float(right)}

In [51]:
def compute15(state):
    a, op, b = state["a"], state["op"], state["b"]
    if op == "+":
        return {"result": a + b}
    elif op == "-":
        return {"result": a - b}
    elif op == "*":
        return {"result": a * b}
    get_stream_writer()(f"dividing {a:g} by {b:g}...")  # new
    return {"result": a / b}


def format15(state):
    return {"answer": f"{state['expression']} = {state['result']:g}"}


def error15(state):
    return {"answer": f"Sorry, I don't know the operator {state['op']!r}."}


def route15(state) -> Literal["compute", "error"]:
    return "compute" if state["op"] in ("+", "-", "*", "/") else "error"


In [52]:
builder = StateGraph(CalcState15)
builder.add_node("parse", parse15)
builder.add_node("compute", compute15)
builder.add_node("format", format15)
builder.add_node("error", error15)
builder.add_edge(START, "parse")
builder.add_conditional_edges("parse", route15)
builder.add_edge("compute", "format")
builder.add_edge("format", END)
builder.add_edge("error", END)
graph15 = builder.compile()

**1.** The custom stream now shows the division message.

In [53]:
for message in graph15.stream({"expression": "10 / 4"}, stream_mode="custom"):
    print(message)

dividing 10 by 4...


**2.** Each `"updates"` piece is `{node_name: changes}`, so we look for the `format` or `error` node and print its
`answer`.

In [54]:
for expression in ["6 * 7", "2 % 3"]:
    for update in graph15.stream({"expression": expression}, stream_mode="updates"):
        for node_name, changes in update.items():
            if node_name in ("format", "error"):
                print(changes["answer"])

6 * 7 = 42
Sorry, I don't know the operator '%'.


## Lesson 16 · Subgraphs

The lesson 16 arithmetic subgraph, rebuilt.

In [55]:
class ArithState(TypedDict):
    expression: str
    a: float
    op: str
    b: float
    result: float


def arith_compute(state: ArithState) -> dict:
    a, op, b = state["a"], state["op"], state["b"]
    if op == "+":
        return {"result": a + b}
    elif op == "-":
        return {"result": a - b}
    elif op == "*":
        return {"result": a * b}
    return {"result": a / b}


def arith_parse(state: ArithState) -> dict:
    left, op, right = state["expression"].split()
    return {"a": float(left), "op": op, "b": float(right)}


arith_builder = StateGraph(ArithState)
arith_builder.add_node("parse", arith_parse)
arith_builder.add_node("compute", arith_compute)
arith_builder.add_edge(START, "parse")
arith_builder.add_edge("parse", "compute")
arith_builder.add_edge("compute", END)
arithmetic = arith_builder.compile()

**1.** Yes. Once `op` is in `AppState`, it's a **shared key**, so it flows back out of the subgraph like `result`
does.

In [56]:
class AppState(TypedDict):
    expression: str
    op: str  # new: now shared with the subgraph
    result: float
    answer: str


def format16(state: AppState) -> dict:
    return {"answer": f"{state['expression']} = {state['result']:g}"}


app_builder = StateGraph(AppState)
app_builder.add_node("arithmetic", arithmetic)
app_builder.add_node("format", format16)
app_builder.add_edge(START, "arithmetic")
app_builder.add_edge("arithmetic", "format")
app_builder.add_edge("format", END)
print(app_builder.compile().invoke({"expression": "3 * 4"}))

{'expression': '3 * 4', 'op': '*', 'result': 12.0, 'answer': '3 * 4 = 12'}


**2.** `times_ten` calls the subgraph a second time from inside a node, building the new expression from the first
result.

In [57]:
class TenState(TypedDict):
    expression: str
    result: float
    times_ten: float


def times_ten(state: TenState) -> dict:
    second = arithmetic.invoke({"expression": f"{state['result']:g} * 10"})
    return {"times_ten": second["result"]}


ten_builder = StateGraph(TenState)
ten_builder.add_node("arithmetic", arithmetic)
ten_builder.add_node("times_ten", times_ten)
ten_builder.add_edge(START, "arithmetic")
ten_builder.add_edge("arithmetic", "times_ten")
ten_builder.add_edge("times_ten", END)
print(ten_builder.compile().invoke({"expression": "3 + 4"}))

{'expression': '3 + 4', 'result': 7.0, 'times_ten': 70.0}


## Lesson 17 · Long-term memory with the store

The lesson 17 graph, rebuilt with its own store.

In [58]:
from dataclasses import dataclass
from typing import NotRequired
from uuid import uuid4

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore


@dataclass
class UserContext:
    user_id: str


class PrefState(TypedDict):
    expression: str
    decimal_places: NotRequired[int]
    result: float
    answer: str

In [59]:
def remember_and_calculate(state: PrefState, runtime: Runtime[UserContext]) -> dict:
    if "decimal_places" in state:
        runtime.store.put(("users", runtime.context.user_id), "prefs", {"decimal_places": state["decimal_places"]})
    left, op, right = state["expression"].split()
    a, b = float(left), float(right)
    if op == "+":
        return {"result": a + b}
    elif op == "-":
        return {"result": a - b}
    elif op == "*":
        return {"result": a * b}
    return {"result": a / b}


def format_and_log(state: PrefState, runtime: Runtime[UserContext]) -> dict:
    prefs = runtime.store.get(("users", runtime.context.user_id), "prefs")
    places = prefs.value["decimal_places"] if prefs else 2
    answer = f"{state['expression']} = {state['result']:.{places}f}"
    runtime.store.put(("users", runtime.context.user_id, "calculations"), str(uuid4()), {"answer": answer})
    return {"answer": answer}

In [60]:
builder = StateGraph(PrefState, context_schema=UserContext)
builder.add_node("calculate", remember_and_calculate)
builder.add_node("format", format_and_log)
builder.add_edge(START, "calculate")
builder.add_edge("calculate", "format")
builder.add_edge("format", END)

store = InMemoryStore()
graph17 = builder.compile(checkpointer=InMemorySaver(), store=store)


def run(thread_id, user_id, inputs):
    config = {"configurable": {"thread_id": thread_id}}
    return graph17.invoke(inputs, config, context={"user_id": user_id})["answer"]

**1.** After the preference is deleted, a new thread falls back to the default of **2** places.

In [61]:
def forget_preferences(user_id: str) -> None:
    store.delete(("users", user_id), "prefs")


print(run("alice-1", "alice", {"expression": "10 / 3", "decimal_places": 4}))
forget_preferences("alice")
print(run("alice-2", "alice", {"expression": "10 / 3"}))

10 / 3 = 3.3333


10 / 3 = 3.33


**2.** Carol's `0` places carry over to her other thread, so `7 / 2` shows as `4` (3.5 rounded to a whole number;
Python rounds a half to the nearest even number). The store has logged **2** calculations for her.

In [62]:
print(run("carol-1", "carol", {"expression": "1 + 1", "decimal_places": 0}))
print(run("carol-2", "carol", {"expression": "7 / 2"}))
print("logged:", len(store.search(("users", "carol", "calculations"))))

1 + 1 = 2
7 / 2 = 4
logged: 2


## Lesson 18 · Tools and `ToolNode`

**1.** A `power` tool. The `ToolMessage` contains `'1024.0'`.

In [63]:
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import tool
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition


@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


@tool
def power(a: float, b: float) -> float:
    """Raise a to the power b."""
    return a ** b

In [64]:
tools18 = [add, multiply, power]

builder = StateGraph(MessagesState)
builder.add_node("tools", ToolNode(tools18))
builder.add_edge(START, "tools")
builder.add_edge("tools", END)
tool_runner = builder.compile()

call = AIMessage(content="", tool_calls=[{"name": "power", "args": {"a": 2, "b": 10}, "id": "call_p", "type": "tool_call"}])
print(tool_runner.invoke({"messages": [call]})["messages"][-1])

content='1024.0' name='power' id='30dcbc95-49f1-4e5c-bbc5-cec68232dfc1' tool_call_id='call_p'


**2.** The first reply asks for two tools at once. `ToolNode` runs both and appends two `ToolMessage`s, so the final
reply reads the last two messages.

In [65]:
def fake_model_two(state: MessagesState) -> dict:
    last = state["messages"][-1]
    if isinstance(last, ToolMessage):
        added, multiplied = state["messages"][-2].content, state["messages"][-1].content
        return {"messages": [AIMessage(content=f"3 + 5 = {added} and 3 * 5 = {multiplied}.")]}
    return {"messages": [AIMessage(content="", tool_calls=[
        {"name": "add", "args": {"a": 3, "b": 5}, "id": "call_a", "type": "tool_call"},
        {"name": "multiply", "args": {"a": 3, "b": 5}, "id": "call_m", "type": "tool_call"},
    ])]}


builder = StateGraph(MessagesState)
builder.add_node("model", fake_model_two)
builder.add_node("tools", ToolNode(tools18))
builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")
final = builder.compile().invoke({"messages": [HumanMessage("Add and multiply 3 and 5.")]})
print(final["messages"][-1].content)

3 + 5 = 8.0 and 3 * 5 = 15.0.


## Lesson 19 · The agent loop with Gemini [LLM]

These answers call Gemini (about 4 calls). Without a key, the cells say so and skip. The exact wording of the replies
can differ from run to run.

In [66]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")
HAS_KEY = bool(os.environ.get("GOOGLE_CLOUD_API_KEY"))
print("API key found:", HAS_KEY)

API key found: True


In [67]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage


@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b


@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b


tools19 = [add, subtract, multiply, divide]
SYSTEM_PROMPT = (
    "You are a calculator assistant. Use the tools for every arithmetic step, "
    "one step at a time. When you have the final result, answer in one short sentence."
)

**1.** It can go either way, because the model decides. In the run saved below, Gemini knew the answer was undefined
and replied straight away, **without** calling `divide`, so there is no `ToolMessage`. If it does call the tool, the
`ToolMessage` from `divide` has `status='error'` and the `ZeroDivisionError` text (thanks to `handle_tool_errors=True`),
and the model reads it and explains in its final message that you can't divide by zero.

In [68]:
if HAS_KEY:
    from llm_setup import get_llm

    llm = get_llm()
    llm_with_tools = llm.bind_tools(tools19)

    def call_model(state: MessagesState) -> dict:
        return {"messages": [llm_with_tools.invoke([SystemMessage(SYSTEM_PROMPT)] + state["messages"])]}

    builder = StateGraph(MessagesState)
    builder.add_node("model", call_model)
    builder.add_node("tools", ToolNode(tools19, handle_tool_errors=True))
    builder.add_edge(START, "model")
    builder.add_conditional_edges("model", tools_condition)
    builder.add_edge("tools", "model")
    out = builder.compile().invoke({"messages": [HumanMessage("What is 10 divided by 0?")]})
    for message in out["messages"]:
        message.pretty_print()
else:
    print("No key: skipping (this answer needs Gemini).")

API key loaded: True


================================ Human Message =================================

What is 10 divided by 0?
================================== Ai Message ==================================

I cannot divide by zero.


**2.** Yes. The thread `quick-1` holds the earlier turn, so "that" means 10 and the answer is 7.

In [69]:
if HAS_KEY:
    quick_agent = create_agent(model=llm, tools=tools19, system_prompt=SYSTEM_PROMPT, checkpointer=InMemorySaver())
    quick = {"configurable": {"thread_id": "quick-1"}}
    quick_agent.invoke({"messages": [HumanMessage("What is 2.5 times 4?")]}, quick)
    reply = quick_agent.invoke({"messages": [HumanMessage("Now subtract 3.")]}, quick)
    print(reply["messages"][-1].text)
else:
    print("No key: skipping (this answer needs Gemini).")

The result is 7.
